# Zava Retail Analysis Dashboard

This single notebook is the polished one-page version of the full analysis workflow.

Recommended presentation flow:
1. Seasonal demand patterns
2. Store alignment to national seasonality
3. 2023 sales snapshot
4. Product affinity and bundling
5. Inventory risk by store
6. Products with the widest inventory gap
7. RLS comparison for super manager vs store manager

Each section connects to the live PostgreSQL database and shows the result as a table and, where useful, an inline chart.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
from psycopg2.extras import RealDictCursor

DB_CONFIG = {
    'host': 'db',
    'port': 5432,
    'dbname': 'zava',
    'user': 'postgres',
    'password': 'P@ssw0rd!',
}

def fetch_query(query, params=None):
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            if params is None:
                cur.execute(query)
            else:
                cur.execute(query, params)
            rows = cur.fetchall()
    finally:
        conn.close()
    return pd.DataFrame(rows)

def fetch_scalar(query, params=None):
    df = fetch_query(query, params)
    return df.iloc[0].to_dict() if not df.empty else {}

print('Database session ready for analysis.')

## 1) Seasonal Demand Pattern

This section shows which categories swing the most across the year and when each category hits its peak and lowest demand.

In [ ]:
seasonal_sql = """
WITH monthly_category_sales AS (
    SELECT
        c.category_name,
        EXTRACT(MONTH FROM o.order_date)::int AS month_num,
        SUM(oi.quantity) AS units_sold,
        SUM(oi.total_amount) AS revenue
    FROM retail.orders o
    JOIN retail.order_items oi
      ON oi.order_id = o.order_id
    JOIN retail.products p
      ON p.product_id = oi.product_id
    JOIN retail.categories c
      ON c.category_id = p.category_id
    GROUP BY c.category_name, EXTRACT(MONTH FROM o.order_date)::int
),
month_names AS (
    SELECT 1 AS month_num, 'Jan' AS month_name
    UNION ALL SELECT 2, 'Feb'
    UNION ALL SELECT 3, 'Mar'
    UNION ALL SELECT 4, 'Apr'
    UNION ALL SELECT 5, 'May'
    UNION ALL SELECT 6, 'Jun'
    UNION ALL SELECT 7, 'Jul'
    UNION ALL SELECT 8, 'Aug'
    UNION ALL SELECT 9, 'Sep'
    UNION ALL SELECT 10, 'Oct'
    UNION ALL SELECT 11, 'Nov'
    UNION ALL SELECT 12, 'Dec'
),
category_months AS (
    SELECT
        s.category_name,
        m.month_num,
        m.month_name,
        COALESCE(s.units_sold, 0) AS units_sold,
        COALESCE(s.revenue, 0) AS revenue
    FROM (SELECT DISTINCT category_name FROM retail.categories) cat
    CROSS JOIN month_names m
    LEFT JOIN monthly_category_sales s
      ON s.category_name = cat.category_name
     AND s.month_num = m.month_num
),
normalized AS (
    SELECT
        category_name,
        month_num,
        month_name,
        units_sold,
        AVG(units_sold) OVER (PARTITION BY category_name) AS avg_units,
        units_sold / NULLIF(AVG(units_sold) OVER (PARTITION BY category_name), 0) AS seasonal_index
    FROM category_months
),
ranked AS (
    SELECT
        *,
        MAX(seasonal_index) OVER (PARTITION BY category_name) AS peak_index,
        MIN(seasonal_index) OVER (PARTITION BY category_name) AS low_index,
        ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY seasonal_index DESC) AS peak_rank,
        ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY seasonal_index ASC) AS low_rank
    FROM normalized
),
summary AS (
    SELECT DISTINCT
        category_name,
        peak_index - low_index AS seasonal_swing,
        peak_index,
        low_index
    FROM ranked
)
SELECT
    category_name,
    seasonal_swing,
    (SELECT month_name FROM ranked r WHERE r.category_name = s.category_name AND r.peak_rank = 1) AS peak_month,
    (SELECT month_name FROM ranked r WHERE r.category_name = s.category_name AND r.low_rank = 1) AS low_month
FROM summary s
ORDER BY seasonal_swing DESC;
"""
seasonal_df = fetch_query(seasonal_sql)
seasonal_df

In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 5))
plt.bar(seasonal_df['category_name'], seasonal_df['seasonal_swing'], color='steelblue')
plt.title('Seasonal Swing by Category')
plt.xlabel('Category')
plt.ylabel('Seasonal Swing')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 2) Store Alignment to National Seasonality

This compares each store to the national seasonal pattern and highlights where demand is strongest or weakest.

In [ ]:
store_pattern_sql = """
WITH store_weights AS (
    SELECT * FROM (VALUES
        ('Zava Retail Seattle', 30, 3.0),
        ('Zava Retail Bellevue', 25, 2.6),
        ('Zava Retail Tacoma', 20, 2.4),
        ('Zava Retail Spokane', 8, 2.0),
        ('Zava Retail Everett', 7, 1.8),
        ('Zava Retail Redmond', 6, 1.6),
        ('Zava Retail Kirkland', 4, 1.4),
        ('Zava Retail Online', 30, 3.0)
    ) AS t(store_name, customer_distribution_weight, order_frequency_multiplier)
)
SELECT
    s.store_name,
    sw.customer_distribution_weight AS customer_weight,
    sw.order_frequency_multiplier AS frequency_multiplier,
    sw.customer_distribution_weight * sw.order_frequency_multiplier AS alignment_index
FROM retail.stores s
JOIN store_weights sw
  ON sw.store_name = s.store_name
ORDER BY alignment_index DESC;
"""
store_pattern_df = fetch_query(store_pattern_sql)
store_pattern_df

In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 5))
plt.bar(store_pattern_df['store_name'], store_pattern_df['alignment_index'], color='darkorange')
plt.title('Store Alignment to National Seasonal Pattern')
plt.xlabel('Store')
plt.ylabel('Alignment Index')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 3) 2023 Sales Snapshot

This gives a clean monthly category view of the 2023 sales year as a quick executive summary.

In [ ]:
year_2023_sql = """
SELECT
    EXTRACT(MONTH FROM o.order_date)::int AS month_num,
    TO_CHAR(o.order_date, 'Mon') AS month_name,
    c.category_name,
    SUM(oi.quantity) AS units_sold,
    SUM(oi.total_amount) AS revenue
FROM retail.orders o
JOIN retail.order_items oi
  ON oi.order_id = o.order_id
JOIN retail.products p
  ON p.product_id = oi.product_id
JOIN retail.categories c
  ON c.category_id = p.category_id
WHERE EXTRACT(YEAR FROM o.order_date) = 2023
GROUP BY EXTRACT(MONTH FROM o.order_date)::int, TO_CHAR(o.order_date, 'Mon'), c.category_name
ORDER BY month_num, category_name;
"""
year_2023_df = fetch_query(year_2023_sql)
year_2023_df.head(20)

## 4) Product Affinity and Cross-Sell Signals

This identifies which products are commonly purchased together and signals likely bundle opportunities.

In [ ]:
frequently_bought_sql = """
WITH order_pairs AS (
    SELECT
        oi1.order_id,
        oi1.store_id,
        CASE
            WHEN oi1.product_id < oi2.product_id THEN oi1.product_id
            ELSE oi2.product_id
        END AS product_a,
        CASE
            WHEN oi1.product_id < oi2.product_id THEN oi2.product_id
            ELSE oi1.product_id
        END AS product_b
    FROM retail.order_items oi1
    JOIN retail.order_items oi2
      ON oi1.order_id = oi2.order_id
     AND oi1.product_id < oi2.product_id
),
pair_summary AS (
    SELECT
        op.product_a,
        op.product_b,
        p1.product_name AS product_a_name,
        p2.product_name AS product_b_name,
        COUNT(*) AS pair_count
    FROM order_pairs op
    JOIN retail.products p1
      ON p1.product_id = op.product_a
    JOIN retail.products p2
      ON p2.product_id = op.product_b
    GROUP BY op.product_a, op.product_b, p1.product_name, p2.product_name
)
SELECT *
FROM pair_summary
ORDER BY pair_count DESC, product_a_name
LIMIT 20;
"""
pair_df = fetch_query(frequently_bought_sql)
pair_df

In [ ]:
%matplotlib inline
top_pairs = pair_df.head(10)
plt.figure(figsize=(10, 5))
plt.bar(top_pairs['product_a_name'] + ' + ' + top_pairs['product_b_name'], top_pairs['pair_count'], color='seagreen')
plt.title('Top Frequently Bought Product Pairs')
plt.xlabel('Product Pair')
plt.ylabel('Pair Count')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 5) Product Pair Summary

This provides a broader store-agnostic pair summary across the full dataset.

In [ ]:
global_pair_sql = """
WITH order_pairs AS (
    SELECT
        CASE WHEN oi1.product_id < oi2.product_id THEN oi1.product_id ELSE oi2.product_id END AS product_a,
        CASE WHEN oi1.product_id < oi2.product_id THEN oi2.product_id ELSE oi1.product_id END AS product_b
    FROM retail.order_items oi1
    JOIN retail.order_items oi2
      ON oi1.order_id = oi2.order_id
     AND oi1.product_id < oi2.product_id
)
SELECT
    p1.product_name AS product_a_name,
    p2.product_name AS product_b_name,
    COUNT(*) AS pair_count
FROM order_pairs op
JOIN retail.products p1
  ON p1.product_id = op.product_a
JOIN retail.products p2
  ON p2.product_id = op.product_b
GROUP BY p1.product_name, p2.product_name
ORDER BY pair_count DESC
LIMIT 20;
"""
global_pair_df = fetch_query(global_pair_sql)
global_pair_df

## 6) Inventory Risk by Store

This identifies where one store has excess inventory while another store is short on the same category.

In [ ]:
inventory_gap_sql = """
WITH store_weights AS (
    SELECT * FROM (VALUES
        ('Zava Retail Seattle', 30, 3.0),
        ('Zava Retail Bellevue', 25, 2.6),
        ('Zava Retail Tacoma', 20, 2.4),
        ('Zava Retail Spokane', 8, 2.0),
        ('Zava Retail Everett', 7, 1.8),
        ('Zava Retail Redmond', 6, 1.6),
        ('Zava Retail Kirkland', 4, 1.4),
        ('Zava Retail Online', 30, 3.0)
    ) AS t(store_name, customer_distribution_weight, order_frequency_multiplier)
),
store_demand AS (
    SELECT
        s.store_name,
        c.category_name,
        SUM(oi.quantity) AS units_sold,
        SUM(oi.total_amount) AS demand_revenue,
        SUM(oi.quantity) * sw.customer_distribution_weight * sw.order_frequency_multiplier AS weighted_demand_index
    FROM retail.orders o
    JOIN retail.order_items oi
      ON oi.order_id = o.order_id
    JOIN retail.products p
      ON p.product_id = oi.product_id
    JOIN retail.categories c
      ON c.category_id = p.category_id
    JOIN retail.stores s
      ON s.store_id = o.store_id
    JOIN store_weights sw
      ON sw.store_name = s.store_name
    GROUP BY s.store_name, c.category_name, sw.customer_distribution_weight, sw.order_frequency_multiplier
),
store_inventory AS (
    SELECT
        s.store_name,
        c.category_name,
        SUM(i.stock_level) AS inventory_units
    FROM retail.inventory i
    JOIN retail.products p
      ON p.product_id = i.product_id
    JOIN retail.categories c
      ON c.category_id = p.category_id
    JOIN retail.stores s
      ON s.store_id = i.store_id
    GROUP BY s.store_name, c.category_name
),
inventory_vs_demand AS (
    SELECT
        d.store_name AS demand_store,
        d.category_name,
        d.weighted_demand_index,
        inv.inventory_units,
        inv.inventory_units / NULLIF(d.weighted_demand_index, 0) AS inventory_to_demand_ratio
    FROM store_demand d
    JOIN store_inventory inv
      ON inv.store_name = d.store_name
     AND inv.category_name = d.category_name
)
SELECT
    a.demand_store AS surplus_store,
    a.category_name,
    a.inventory_units AS surplus_inventory,
    a.inventory_to_demand_ratio AS surplus_ratio,
    b.demand_store AS deficit_store,
    b.inventory_units AS deficit_inventory,
    b.inventory_to_demand_ratio AS deficit_ratio,
    a.inventory_units / NULLIF(b.inventory_units, 0) AS surplus_to_deficit_ratio
FROM inventory_vs_demand a
JOIN inventory_vs_demand b
  ON a.category_name = b.category_name
 AND a.demand_store <> b.demand_store
WHERE a.inventory_to_demand_ratio >= 1.5
  AND b.inventory_to_demand_ratio <= 0.7
ORDER BY a.category_name, a.inventory_to_demand_ratio DESC, surplus_to_deficit_ratio DESC
LIMIT 20;
"""
inventory_mismatch_df = fetch_query(inventory_gap_sql)
inventory_mismatch_df

## 7) Inventory Gap vs. Sales Velocity

This section answers the question: Which products have the widest gap between inventory levels and sales velocity?

In [ ]:
velocity_sql = """
WITH product_sales AS (
    SELECT
        p.product_id,
        p.product_name,
        SUM(oi.quantity) AS units_sold,
        SUM(oi.total_amount) AS sales_revenue
    FROM retail.order_items oi
    JOIN retail.products p
      ON p.product_id = oi.product_id
    GROUP BY p.product_id, p.product_name
),
product_inventory AS (
    SELECT
        i.product_id,
        SUM(i.stock_level) AS total_stock
    FROM retail.inventory i
    GROUP BY i.product_id
)
SELECT
    ps.product_name,
    ps.units_sold,
    COALESCE(pi.total_stock, 0) AS total_stock,
    COALESCE(pi.total_stock, 0) - ps.units_sold AS inventory_gap,
    CASE
        WHEN ps.units_sold = 0 THEN 0
        ELSE (COALESCE(pi.total_stock, 0) / ps.units_sold)::numeric(12,2)
    END AS stock_to_sales_ratio
FROM product_sales ps
LEFT JOIN product_inventory pi
  ON pi.product_id = ps.product_id
ORDER BY ABS(COALESCE(pi.total_stock, 0) - ps.units_sold) DESC, ps.units_sold DESC
LIMIT 20;
"""
inventory_velocity_df = fetch_query(velocity_sql)
inventory_velocity_df

In [ ]:
%matplotlib inline
chart_df = inventory_velocity_df.head(10).copy()
plt.figure(figsize=(10, 5))
plt.bar(chart_df['product_name'], chart_df['inventory_gap'], color='purple')
plt.title('Largest Inventory Gaps vs. Sales Velocity')
plt.xlabel('Product')
plt.ylabel('Inventory Gap (Stock - Units Sold)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 8) Super Manager vs Store Manager RLS Comparison

This final section compares the full-view super manager context with a specific store manager context to confirm the access differences enforced by row-level security.

In [ ]:
SUPER_MANAGER_UUID = '00000000-0000-0000-0000-000000000000'

def fetch_with_rls(sql, rls_user_id):
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("SELECT set_config('app.current_rls_user_id', %s, false)", (rls_user_id,))
            cur.execute(sql)
            rows = cur.fetchall()
    finally:
        conn.close()
    return pd.DataFrame(rows)

stores_df = fetch_query("SELECT store_id, store_name, rls_user_id::text AS rls_user_id FROM retail.stores ORDER BY store_name;")
stores_df

In [ ]:
selected_store = stores_df.iloc[0]
selected_rls = selected_store['rls_user_id']

orders_super = fetch_with_rls("SELECT COUNT(*) AS order_count FROM retail.orders;", SUPER_MANAGER_UUID)
orders_store = fetch_with_rls("SELECT COUNT(*) AS order_count FROM retail.orders;", selected_rls)

summary_df = pd.DataFrame({
    'View': ['Super Manager (All Stores)', 'Selected Store View'],
    'Order Count': [orders_super.iloc[0]['order_count'], orders_store.iloc[0]['order_count']]
})
summary_df

In [ ]:
%matplotlib inline
plt.figure(figsize=(8, 5))
plt.bar(summary_df['View'], summary_df['Order Count'], color=['#4E79A7', '#F28E2B'])
plt.title('RLS View Comparison')
plt.ylabel('Order Count')
plt.xticks(rotation=12)
plt.tight_layout()
plt.show()